# Name That LLM | Part Two + Official Spectral Helper

Hypothesis: the published eight-head readout bank adds author-style evidence to the newer Part Two baseline. Keep its fixed main weights, centered twins and average-linkage groups. Add **10% `official_only` helper** to **90% raw Part Two probabilities**, then apply one final 300-step Sinkhorn to the selected candidate.

The two notebooks have the same models, inputs, feature extraction and mixing. Only the pretrained helper bank differs: 2699 official rows versus 2699 official + 4404 synthetic rows. No new model or classifier training is performed here. Current test features are always inferred; saved hidden-test answers are never used.

This adds one truncated Qwen3-8B LoRA24 backbone to Part Two. There is no Qwen3.5, ATLAS or large complementary ensemble. Unlike iterations 11–12, these notebooks do not repeat official-training feature extraction; they run only three training anchors for extractor parity. Runtime and score are unmeasured; improvement is not guaranteed.

## Credits and scope

Main branch: **Nishant Dahal**, [Two-stage LoRA, part two](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two), source-reported LB 0.620. Public helper and readout implementation: **kirilltym**, [Teach More, Keep the Signature](https://www.kaggle.com/code/kirilltym/0-639-teach-more-keep-the-signature?scriptVersionId=361634630) and [spectral/style heads](https://www.kaggle.com/datasets/kirilltym/ntl-spectral-style-heads-paired-4404). Verified legacy archive: **fleongg**, [NTL Legacy Qwen24 Recovery](https://www.kaggle.com/datasets/fleongg/ntl-legacy-qwen24-recovery). Original public LoRA24: **Vladislav Lavrentev / lightningv08**, retained in the recovery manifest and adapter README. Upstream adaptive centering and model work: **Lucas Boesen**. Additional pool/model credits are retained below.

The small third-party `readouts.py` is executed through `PublicReadoutBank` after a pinned SHA256 check. Its original functions are preserved; all notebook orchestration and our model adapters use classes. No raw synthetic passages are required or included. Retain the licenses and notices of every input; this adaptation does not relicense them.

These are **new ablations**, not reproductions of the complete 0.639 stack: our parent is Part Two alone, without the original complementary ensemble and ATLAS. A 10% helper that helps that full parent may behave differently here. No local full-ensemble OOF evidence or leaderboard improvement is claimed.


## Seven versioned inputs, embedded in Kaggle metadata

Both notebooks need the same inputs. Only the selected bank changes. Confirm seven inputs in Kaggle after import; existing editor sessions may keep their old attachments. These IDs and public file manifests were checked through the Kaggle API on 2026-10-25.

| Section | Input | Version |
|---|---|---|
| Competition | [Name That LLM](https://www.kaggle.com/competitions/name-that-llm) | Competition data |
| Dataset | [NTL two-stage LoRA part two](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend) | 1, datasetVersion 21119693 |
| Model | [Qwen3-14B-Base](https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b-base/1) | 1 |
| Model | [Qwen2.5-14B](https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/14b/1) | 1 |
| Dataset | [NTL Legacy Qwen24 Recovery](https://www.kaggle.com/datasets/fleongg/ntl-legacy-qwen24-recovery) | datasetVersion 21066826 |
| Dataset | [NTL Spectral Style Heads — Paired 4404](https://www.kaggle.com/datasets/kirilltym/ntl-spectral-style-heads-paired-4404) | datasetVersion 21156661 |
| Model | [Qwen3-8B-Base](https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/8b-base/1) | 1 |

Use **GPU T4 x2**, Internet **off**, and a fresh session. No private inputs, ATLAS, Qwen3.5, notebook outputs, or full complementary stack are required. Qwen2.5-7B cannot replace Qwen2.5-14B. The small recovery dataset replaces the inaccessible original training notebook and the large historical archive.

For CPU-only preflight, select accelerator None and set `config.run_inference = False` before the preflight cell. Metadata, SHA256, tokenizer IDs, sklearn-bank prediction on saved anchors and a real tiny dense-LoRA CPU construction/forward pass are checked before expensive inference. This cannot establish GPU parity. Preflight restores about 130 MiB of public checkpoint files to scratch. A fresh session is required on rerun.

The pretrained banks were saved with scikit-learn 1.6.1. Version warnings are recorded, and an actual CPU prediction probe runs before GPU work. That probe catches execution incompatibility; it does not establish numerical equivalence across library versions. The notebook does not install packages. Use a compatible Kaggle environment rather than bypassing a failed check.


The ordinary floating-point LoRA adapter does not require torchao. During adapter construction only, its availability checks in PEFT and its imported dispatch aliases are disabled, then restored. No package installation, quantization or model-weight change is introduced.


In [ ]:
'''1. Offline environment
The source used Kaggle image sha256:9a6bcfda6d03b2cd9aa347ee8c1bae62043328ed0f4afe9bda014e4f8bbf9435.
Use its package environment where available; this notebook does not install or upgrade libraries.
No three-row shortcut: every test row receives real inference. A visible run is not a hidden runtime benchmark.'''

In [ ]:
import os
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[key] = '4'
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')

In [ ]:
import gc
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import struct
import tempfile
import time
from contextlib import contextmanager
import numpy as np
import pandas as pd
import torch
from torch import nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import hstack
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, RidgeClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits
import importlib.util
import warnings
import joblib
from sklearn.exceptions import InconsistentVersionWarning

In [ ]:
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
LORAS = {'lora_qwen3_14b': dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type='qwen3', hidden_size=5120, num_hidden_layers=40)), 'lora_qwen25_14b': dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type='qwen2', hidden_size=5120, num_hidden_layers=48))}
SMALL_WORDS = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.0), ('word', 1, 3, 180000, 0.7), ('shape', 2, 6, 60000, 0.5), ('small', 1, 3, 50000, 0.5)]
INPUT_URLS = {'competition': 'https://www.kaggle.com/competitions/name-that-llm', 'assets': 'https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend', 'lora_qwen3_14b': 'https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/14b-base/1', 'lora_qwen25_14b': 'https://www.kaggle.com/models/qwen-lm/qwen2.5/transformers/14b/1'}
PINNED_SHA256 = {'lora_qwen3_14b': {'meta.json': '293119872ff3f170ec85e8be02c80916e39048213de31cb45f8b84a2cf74d775', 'heads.safetensors': 'e1d9c1ac833a1cc780f1423341bb5299826271f5aeed2c40d3ec193633d45faf'}, 'lora_qwen25_14b': {'meta.json': '1d04028b7fe22beda04f7125d09dd5bc9bee9c8524f721c0aa7bc61379fad6fc', 'heads.safetensors': '73762ab515ed984efc9189063ee4efddd1a0b0b2620aab6420b3e56e1a1c752e'}, 'deberta_v3_large': {'meta.json': 'deb3c891d6172018ec06b7b3de33f9a13ac6c44d6bf8a9725b53ce8c7c93a546', 'head.safetensors': '43db1fbe725abfc2299e6618f2f6b3e7cb7a63e3981cb4325dd6961e6e1afb06'}}
EXPECTED_ADAPTER_BYTES = {'lora_qwen3_14b': 173456688, 'lora_qwen25_14b': 183558248}
BASE_PATHS = {'lora_qwen3_14b': 'qwen-lm/qwen-3/transformers/14b-base/1', 'lora_qwen25_14b': 'qwen-lm/qwen2.5/transformers/14b/1'}
SOURCE_URL = 'https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two'
SOURCE_SHA256 = '6a33ee530a46129746e8b06588829e50fc13a67639f019d378ffd44ff963dee1'
HELPER_BANK_URL = 'https://www.kaggle.com/datasets/kirilltym/ntl-spectral-style-heads-paired-4404'
HELPER_ARCHIVE_URL = 'https://www.kaggle.com/datasets/fleongg/ntl-legacy-qwen24-recovery'
HELPER_BASE_URL = 'https://www.kaggle.com/models/qwen-lm/qwen-3/transformers/8b-base/1'
HELPER_LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
HELPER_HEAD_WEIGHTS = dict(rbf=0.05, fourier=0.075, laplace=0.075, knn=0.05, centered=0.10, word_pca=0.15, char_pca=0.20, late=0.30)
HELPER_ADAPTER_SHA256 = 'a6aff409ae2df897717875373d038635fe5012fbf93557c90aea3457fda0e7f8'
HELPER_HEADS_SHA256 = '2176ade29e6eb97bd3a6c204f944959f9757a80c6f1ff475898c23bce60ecd96'
LEGACY_MANIFEST_SHA256 = '010c41fd5f7f6227573cd6f7d27b93e49ce433e67c70003c9a55cf971fd5ef8d'
BANK_MANIFEST_SHA256 = '3ff9a81eb3c430020fb65d32eee11a1634fb285bdccd12997da61e0567eabdd0'
READOUT_SOURCE_SHA256 = '2f420e8dd1da0e08d34fd503d912402a9043788946bf287925dbf69a26a43bf4'

HELPER_CONDITION = 'official_only'

In [ ]:
'''2. Configuration
Helper weight is fixed at 10%; only HELPER_CONDITION differs between iterations 13 and 14.
No automatic component fallback, training-cache loading or repeated official-training inference.'''

In [ ]:
class ExperimentConfig:
    def __init__(self):
        self.run_inference = True
        self.input_root = Path('/kaggle/input')
        self.output_root = Path('/kaggle/working')
        self.scratch_root = Path('/kaggle/temp/ntl_public_helper')
        self.labels = list(LABELS)
        self.loras = LORAS
        self.helper_condition = HELPER_CONDITION
        self.helper_weight = 0.10


In [ ]:
config = ExperimentConfig()
'''Set False for CPU-only input checks; this produces no submission.'''
config.run_inference = True

In [ ]:
'''3. Exact inputs and structural checks
Inspect the printed EOS and resolved model paths. Config-only folders and missing shards are rejected.
Metadata and head checksums pin the reviewed assets; base model weights are structurally checked, not fully hashed.'''

In [ ]:
class FileChecks:
    @staticmethod
    def sha256(path):
        with Path(path).open('rb') as stream:
            return hashlib.file_digest(stream, 'sha256').hexdigest()

    @staticmethod
    def tensor_shapes(path):
        with Path(path).open('rb') as stream:
            prefix = stream.read(8)
            if len(prefix) != 8:
                raise ValueError(f'Invalid safetensors header: {path}')
            size = struct.unpack('<Q', prefix)[0]
            if not 2 <= size <= 16 * 1024 * 1024:
                raise ValueError(f'Unexpected safetensors header size: {path}')
            header = json.loads(stream.read(size))
        return {key: value['shape'] for key, value in header.items() if key != '__metadata__'}

    @staticmethod
    def weight_files(directory):
        directory = Path(directory)
        for name in ('model.safetensors', 'pytorch_model.bin'):
            path = directory / name
            if path.is_file() and path.stat().st_size > 0:
                return [path]
        for name in ('model.safetensors.index.json', 'pytorch_model.bin.index.json'):
            index = directory / name
            if index.is_file():
                mapping = json.loads(index.read_text())['weight_map']
                paths = [directory / part for part in sorted(set(mapping.values()))]
                if paths and all(path.is_file() and path.stat().st_size > 0 for path in paths):
                    return paths
        raise FileNotFoundError(f'No complete model weights in {directory}; config-only folders are not checkpoints')

In [ ]:
class InputAssets:
    def __init__(self, config):
        self.config = config
        self.assets = None
        self.bases = {}
        self.manifest = {}

    def resolve_base(self, key, spec):
        relative = BASE_PATHS[key]
        roots = [self.config.input_root / 'models' / relative, self.config.input_root / relative,
                 self.config.input_root / Path(*Path(relative).parts[1:])]
        candidates = [root for root in roots if (root / 'config.json').is_file()]
        if not candidates:
            raise FileNotFoundError(f'Attach the exact model: {INPUT_URLS[key]}. Expected {relative}')
        if len(candidates) != 1:
            raise ValueError(f'Multiple mounted copies of {key}; attach only the requested model revision')
        path = candidates[0]
        raw = json.loads((path / 'config.json').read_text())
        if any(raw.get(field) != value for field, value in spec.items()):
            raise ValueError(f'Wrong architecture for {key}: {path}')
        files = FileChecks.weight_files(path)
        tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        meta = json.loads((self.assets / key / 'meta.json').read_text())
        eos = meta['cfg']['eos_id']
        if not 0 <= eos < len(tokenizer):
            raise ValueError(f'Training EOS {eos} is outside the tokenizer vocabulary for {key}')
        self.manifest[key] = dict(base=str(path), config_sha256=FileChecks.sha256(path / 'config.json'),
            training_eos_id=eos, tokenizer_eos_id=tokenizer.eos_token_id,
            weights=[dict(name=f.name, bytes=f.stat().st_size) for f in files])
        return path

    def validate_lora(self, key, spec):
        directory = self.assets / key
        for relative in ['meta.json', 'heads.safetensors', 'adapter/adapter_config.json', 'adapter/adapter_model.safetensors']:
            path = directory / relative
            if not path.is_file() or path.stat().st_size == 0:
                raise FileNotFoundError(f'Missing {path}. Attach {INPUT_URLS["assets"]}')
        for filename, expected in PINNED_SHA256[key].items():
            if FileChecks.sha256(directory / filename) != expected:
                raise ValueError(f'{key}/{filename} differs from the verified Part Two asset; review the dataset version')
        meta = json.loads((directory / 'meta.json').read_text())
        if meta['labels'] != self.config.labels:
            raise ValueError(f'Checkpoint label order mismatch: {key}')
        settings = meta['cfg']
        if settings.get('eos_id') != 151643 or not 0 < settings['keep'] <= spec['base']['num_hidden_layers']:
            raise ValueError(f'Unexpected training EOS or retained layer count: {key}')
        if len(settings['layers']) != 8 or min(settings['layers']) < 1 or max(settings['layers']) > settings['keep']:
            raise ValueError(f'Invalid pooling layers in {key}')
        adapter = json.loads((directory / 'adapter/adapter_config.json').read_text())
        if adapter['r'] != settings['r'] or adapter['lora_alpha'] != settings['alpha']:
            raise ValueError(f'Adapter and training metadata differ: {key}')
        hidden = spec['base']['hidden_size']
        shapes = FileChecks.tensor_shapes(directory / 'heads.safetensors')
        for prefix in ['plain', 'centered']:
            if shapes.get(prefix + '.weight') != [len(self.config.labels), hidden * 8] or shapes.get(prefix + '.bias') != [len(self.config.labels)]:
                raise ValueError(f'Invalid {prefix} head dimensions for {key}')
        for layer in range(8):
            for suffix in ['weight', 'bias']:
                if shapes.get(f'norms.{layer}.{suffix}') != [hidden]:
                    raise ValueError(f'Invalid pooling normalization for {key}')
        for tensor, shape in FileChecks.tensor_shapes(directory / 'adapter/adapter_model.safetensors').items():
            if not shape or settings['r'] not in shape:
                raise ValueError(f'Unexpected LoRA tensor shape: {key}/{tensor}')
        if (directory / 'adapter/adapter_model.safetensors').stat().st_size != EXPECTED_ADAPTER_BYTES[key]:
            raise ValueError(f'Unexpected adapter file size for {key}; check the dataset version')

    def preflight(self):
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        roots = [self.config.input_root / 'datasets/nishantdahal/ntl-handwriting-blend',
                 self.config.input_root / 'ntl-handwriting-blend']
        candidates = [root for root in roots if (root / 'pool_v4.parquet').is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(f'Attach one copy of {INPUT_URLS["assets"]}; old ntl-two-stage-lora assets are incompatible')
        self.assets = candidates[0]
        for key, spec in self.config.loras.items():
            self.validate_lora(key, spec)
            self.bases[key] = self.resolve_base(key, spec['base'])
        deb = self.assets / 'deberta_v3_large'
        for name, expected in PINNED_SHA256['deberta_v3_large'].items():
            if not (deb / name).is_file() or FileChecks.sha256(deb / name) != expected:
                raise ValueError(f'Missing or incompatible DeBERTa asset: {name}')
        meta = json.loads((deb / 'meta.json').read_text())
        if meta['labels'] != self.config.labels or meta['max_len'] != 384:
            raise ValueError('DeBERTa label order or token limit differs from the source')
        if FileChecks.tensor_shapes(deb / 'head.safetensors') != {'bias': [18], 'weight': [18, 1024]}:
            raise ValueError('Invalid DeBERTa head')
        FileChecks.weight_files(deb / 'encoder')
        AutoTokenizer.from_pretrained(str(deb / 'encoder'), local_files_only=True)
        self.manifest['adapter_sha256'] = {key: FileChecks.sha256(self.assets / key / 'adapter/adapter_model.safetensors')
                                           for key in self.config.loras}
        self.manifest['asset_sha256'] = PINNED_SHA256
        self.manifest['source_code_sha256'] = SOURCE_SHA256
        (self.config.output_root / 'input_manifest.json').write_text(json.dumps(self.manifest, indent=2))
        print('INPUT CHECKS PASSED: exact Part Two assets, complete base weights, 18 labels, training EOS 151643', flush=True)
        return self

    def load_data(self):
        roots = [self.config.input_root / 'competitions/name-that-llm', self.config.input_root / 'name-that-llm']
        candidates = [root for root in roots if (root / 'train.csv').is_file() and (root / 'test.csv').is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(f'Attach the competition data: {INPUT_URLS["competition"]}')
        directory = candidates[0]
        train = pd.read_csv(directory / 'train.csv', dtype={'id': str})
        test = pd.read_csv(directory / 'test.csv', dtype={'id': str})
        if not {'id', 'text', 'model', 'prompt_id'}.issubset(train) or not {'id', 'text'}.issubset(test):
            raise ValueError('Unexpected competition CSV schema')
        for frame in [train, test]:
            if frame.empty or frame['id'].isna().any() or frame['id'].duplicated().any():
                raise ValueError('Expected nonempty CSVs with unique non-null IDs')
        if set(train['model']) != set(self.config.labels) or train[['text', 'prompt_id']].isna().any().any():
            raise ValueError('Missing training text, prompt IDs or author labels')
        test['text'] = test['text'].fillna('').astype(str)
        pool = pd.read_parquet(self.assets / 'pool_v4.parquet')
        if not {'text', 'model'}.issubset(pool) or len(pool) != 75783:
            raise ValueError('Expected the verified 75,783-row pool_v4; check the dataset version')
        if pool[['text', 'model']].isna().any().any() or not pool['model'].isin(self.config.labels).all():
            raise ValueError('Invalid public-pool text or author labels')
        self.data_hashes = {name: FileChecks.sha256(directory / f'{name}.csv') for name in ['train', 'test']}
        return train, test, pool

    def feature_signature(self):
        record = dict(train_sha256=self.data_hashes['train'], labels=self.config.labels, assets=PINNED_SHA256,
                      adapters=self.manifest['adapter_sha256'],
                      bases={key: {field: value for field, value in record.items() if field != 'base'}
                             for key, record in self.manifest.items() if key in self.config.loras})
        return hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()

In [ ]:
'''4. Timing, local staging and source probability calibration'''

In [ ]:
class StageTimer:
    def __init__(self, root):
        self.path = root / 'stage_timings.json'
        self.records = []

    @contextmanager
    def measure(self, name):
        record = {'stage': name, 'status': 'running'}
        self.records.append(record)
        self.path.write_text(json.dumps(self.records, indent=2))
        print(f'START {name}', flush=True)
        started = time.monotonic()
        try:
            yield
        except BaseException:
            record['status'] = 'failed'
            raise
        else:
            record['status'] = 'complete'
        finally:
            record['seconds'] = time.monotonic() - started
            self.path.write_text(json.dumps(self.records, indent=2))
            print(f"END {name}: {record['status']}, {record['seconds'] / 60:.1f} min", flush=True)

In [ ]:
class LocalCheckpointStage:
    @staticmethod
    def copy(source, target):
        source, target = Path(source), Path(target)
        if target.exists():
            raise FileExistsError(f'Use a fresh session; scratch checkpoint already exists: {target}')
        files = [path for path in source.rglob('*') if path.is_file()]
        target.parent.mkdir(parents=True, exist_ok=True)
        total = sum(path.stat().st_size for path in files)
        if shutil.disk_usage(target.parent).free < total + 2**30:
            raise OSError('Insufficient scratch space to stage checkpoint before GPU loading')
        for path in files:
            output = target / path.relative_to(source)
            output.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, output)
            if output.stat().st_size != path.stat().st_size:
                raise OSError(f'Incomplete staged checkpoint file: {output}')
        print(f'Staged {total / 2**20:.0f} MiB: {source.name}', flush=True)
        return target

In [ ]:
class ProbabilityMath:
    @staticmethod
    def softmax(values):
        exponential = np.exp(values - values.max(1, keepdims=True))
        return exponential / exponential.sum(1, keepdims=True)

    @staticmethod
    def check(probabilities, rows):
        if probabilities.shape != (rows, len(LABELS)) or not np.isfinite(probabilities).all() or (probabilities < 0).any():
            raise ValueError('Invalid probability matrix')
        if not np.allclose(probabilities.sum(1), 1, atol=1e-5):
            raise ValueError('Probability rows must sum to one')

    @staticmethod
    def balance(probabilities, iters=300):
        probabilities = probabilities.copy()
        if len(probabilities) < 50:
            return probabilities
        scales = np.ones(len(LABELS))
        target = np.full(len(LABELS), len(probabilities) / len(LABELS))
        for _ in range(iters):
            result = probabilities * scales
            result /= result.sum(1, keepdims=True)
            scales *= target / result.sum(0)
        result = probabilities * scales
        return result / result.sum(1, keepdims=True)

In [ ]:
class NotebookContractChecks:
    @staticmethod
    def run():
        rng = np.random.RandomState(42)
        probabilities = ProbabilityMath.softmax(rng.normal(size=(90, len(LABELS))))
        balanced = ProbabilityMath.balance(probabilities)
        ProbabilityMath.check(balanced, 90)
        np.testing.assert_allclose(balanced.sum(0), np.full(len(LABELS), 90 / len(LABELS)), atol=1e-5)
        np.testing.assert_array_equal(ProbabilityMath.balance(probabilities[:3]), probabilities[:3])
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory)
            (path / 'config.json').write_text('{}')
            try:
                FileChecks.weight_files(path)
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('A config-only folder was accepted')
            (path / 'model.safetensors.index.json').write_text(json.dumps({'weight_map': {'a': 'one', 'b': 'two'}}))
            (path / 'one').write_bytes(b'fixture')
            try:
                FileChecks.weight_files(path)
            except FileNotFoundError:
                pass
            else:
                raise AssertionError('An incomplete sharded checkpoint was accepted')
            (path / 'two').write_bytes(b'fixture')
            assert len(FileChecks.weight_files(path)) == 2

NotebookContractChecks.run()


In [ ]:
'''5. Run input checks before any expensive model inference'''

In [ ]:
class LegacyArchive:
    def __init__(self, root, scratch):
        self.root, self.scratch = root, scratch

    def restore(self):
        manifest_path = self.root / 'legacy_recovery_manifest.json'
        if FileChecks.sha256(manifest_path) != LEGACY_MANIFEST_SHA256:
            raise ValueError('Legacy archive manifest differs from the verified public recovery version')
        manifest = json.loads(manifest_path.read_text())
        if not manifest['independently_verified_published_migration_manifest']:
            raise ValueError('Unverified legacy archive')
        target = self.scratch / 'legacy_lora24'
        if target.exists():
            raise FileExistsError('Start a fresh session; restored legacy checkpoint already exists')
        target.parent.mkdir(parents=True, exist_ok=True)
        required_bytes = sum(record['bytes'] for record in manifest['files'].values())
        if shutil.disk_usage(target.parent).free < required_bytes + 2**30:
            raise OSError('Insufficient scratch space for the verified legacy checkpoint')
        for name, record in manifest['files'].items():
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts or Path(record['flat_file']).name != record['flat_file']:
                raise ValueError('Invalid relative path in the legacy manifest')
            source = self.root / record['flat_file']
            if not source.is_file() or source.stat().st_size != record['bytes'] or FileChecks.sha256(source) != record['sha256']:
                raise ValueError(f'Missing or corrupted legacy archive file: {source.name}')
            destination = target / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, destination)
            if FileChecks.sha256(destination) != record['sha256']:
                raise OSError(f'Legacy checkpoint copy checksum mismatch: {relative}')
        self.manifest = manifest
        return target

In [ ]:
class PublicReadoutBank:
    def __init__(self, root, condition, labels):
        if condition not in {'official_only', 'synthetic4404'}:
            raise ValueError('Unknown public readout bank')
        if FileChecks.sha256(root / 'COMPLETE.json') != BANK_MANIFEST_SHA256:
            raise ValueError('The public readout manifest version changed')
        if FileChecks.sha256(root / 'readouts.py') != READOUT_SOURCE_SHA256:
            raise ValueError('The credited readout implementation changed')
        manifest = json.loads((root / 'COMPLETE.json').read_text())
        self.record = manifest['banks'][condition]
        expected_rows = 0 if condition == 'official_only' else 4404
        if not manifest['full_fit'] or manifest['adapter_training'] or self.record['official_rows'] != 2699 or self.record['synthetic_rows'] != expected_rows:
            raise ValueError('Unexpected public bank training condition')
        path = root / f'{condition}.joblib'
        if FileChecks.sha256(path) != self.record['readout_sha256']:
            raise ValueError(f'Corrupted or incorrect public bank: {condition}')
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', InconsistentVersionWarning)
            self.bank = joblib.load(path)
        self.compatibility_warnings = sorted({str(item.message) for item in caught})
        if self.compatibility_warnings:
            print(f'Public bank: {len(self.compatibility_warnings)} sklearn version warnings; running the CPU predict probe before GPU work.', flush=True)
        if self.bank['layers'] != HELPER_LAYERS or self.bank['adapter_sha256'] != HELPER_ADAPTER_SHA256:
            raise ValueError('Public bank requires a different LoRA adapter or pooling layout')
        if self.bank['official_rows'] != 2699 or self.bank['synthetic_rows'] != expected_rows:
            raise ValueError('Bank contents do not match its training manifest')
        bank_labels = self.bank['labels']
        if len(bank_labels) != len(labels) or set(bank_labels) != set(labels):
            raise ValueError('The helper does not contain the same 18 author labels')
        if self.bank['head_weights'] != HELPER_HEAD_WEIGHTS or set(self.bank['heads']) != set(HELPER_HEAD_WEIGHTS):
            raise ValueError('Unexpected public readout head set or weights')
        self.columns = [bank_labels.index(label) for label in labels]
        namespace = {'__name__': 'credited_public_readouts'}
        exec(compile((root / 'readouts.py').read_text(), 'credited_public_readouts', 'exec'), namespace)
        self.predict_bank = namespace['predict_bank']
        self.condition = condition
        self.manifest = manifest

    def predict(self, features, texts):
        if features.shape != (len(texts), 32768) or not np.isfinite(features).all():
            raise ValueError('Expected the exact 8 x 4096 legacy LoRA24 feature layout')
        probabilities, heads = self.predict_bank(self.bank, features, list(texts))
        ProbabilityMath.check(probabilities, len(texts))
        aligned = probabilities[:, self.columns]
        ProbabilityMath.check(aligned, len(texts))
        return aligned, {name: values[:, self.columns] for name, values in heads.items()}

In [ ]:
class HelperInputs(InputAssets):
    @staticmethod
    def unique_root(input_root, relative, marker):
        roots = [input_root / 'datasets' / relative, input_root / Path(relative).name]
        found = [root for root in roots if (root / marker).is_file()]
        if len(found) != 1:
            raise FileNotFoundError(f'Attach one verified copy of https://www.kaggle.com/datasets/{relative}')
        return found[0]

    def prepare_helper(self, train):
        archive_root = self.unique_root(self.config.input_root, 'fleongg/ntl-legacy-qwen24-recovery', 'legacy_recovery_manifest.json')
        bank_root = self.unique_root(self.config.input_root, 'kirilltym/ntl-spectral-style-heads-paired-4404', 'night24_readout_banks/COMPLETE.json') / 'night24_readout_banks'
        self.legacy = LegacyArchive(archive_root, self.config.scratch_root).restore()
        self.public_bank = PublicReadoutBank(bank_root, self.config.helper_condition, self.config.labels)
        expected_train = self.public_bank.manifest['official_train_sha256']
        if self.data_hashes['train'] != expected_train:
            raise ValueError('Official train.csv differs from the public bank and anchor training data')
        relative = Path('qwen-lm/qwen-3/transformers/8b-base/1')
        roots = [self.config.input_root / 'models' / relative, self.config.input_root / relative,
                 self.config.input_root / Path(*relative.parts[1:])]
        found = [root for root in roots if (root / 'config.json').is_file()]
        if len(found) != 1:
            raise FileNotFoundError('Attach Qwen3-8B-Base Transformers version 1: ' + HELPER_BASE_URL)
        self.helper_base = found[0]
        configuration = json.loads((self.helper_base / 'config.json').read_text())
        if any(configuration.get(key) != value for key, value in {'model_type': 'qwen3', 'hidden_size': 4096, 'num_hidden_layers': 36}.items()):
            raise ValueError('The helper base must be the full Qwen3-8B-Base, not a config-only or truncated checkpoint')
        FileChecks.weight_files(self.helper_base)
        if FileChecks.sha256(self.legacy / 'adapter/adapter_model.safetensors') != HELPER_ADAPTER_SHA256:
            raise ValueError('The legacy adapter does not match the bank feature extractor')
        if FileChecks.sha256(self.legacy / 'heads.safetensors') != HELPER_HEADS_SHA256:
            raise ValueError('The legacy pooling normalizers differ from the bank extractor')
        with np.load(self.legacy / 'readouts.npz', allow_pickle=False) as archive:
            if archive['pool_layers'].tolist() != HELPER_LAYERS or archive['classes'].tolist() != self.public_bank.bank['labels']:
                raise ValueError('Legacy label order or pooling layers differ from the bank')
        tokenizer = AutoTokenizer.from_pretrained(str(self.legacy / 'tokenizer'), local_files_only=True)
        if tokenizer.eos_token_id != 151645:
            raise ValueError('Legacy helper requires its archived tokenizer EOS 151645, not Part Two EOS 151643')
        self.anchors = np.load(self.legacy / 'anchor_features.npz', allow_pickle=False)
        if self.anchors['train_sha256'].item() != expected_train or self.anchors['features'].shape != (3, 32768):
            raise ValueError('Wrong legacy anchor features')
        ids = self.anchors['ids'].astype(str).tolist()
        indexed = train.set_index('id')
        if not set(ids).issubset(indexed.index):
            raise ValueError('Anchor IDs are missing from the official train CSV')
        self.anchor_texts = indexed.loc[ids, 'text'].tolist()
        with threadpool_limits(limits=4):
            self.public_bank.predict(self.anchors['features'], self.anchor_texts)
        LegacyLoRAFeatures.probe_dense_peft()
        report = dict(condition=self.config.helper_condition, adapter_sha256=HELPER_ADAPTER_SHA256,
            pooling_layers=HELPER_LAYERS, retained_layers=24, feature_width=32768,
            legacy_eos=151645, part_two_eos=151643, final_backbone_norm='preserved',
            bank_sha256=self.public_bank.record['readout_sha256'],
            source_sha256=READOUT_SOURCE_SHA256, helper_base=str(self.helper_base),
            sklearn_version_warnings=self.public_bank.compatibility_warnings,
            cpu_readout_probe_passed=True, cpu_dense_lora_probe_passed=True, gpu_anchor_probe_passed=False)
        (self.config.output_root / 'helper_input_manifest.json').write_text(json.dumps(report, indent=2))
        print('HELPER INPUT CHECKS PASSED: public bank, exact legacy adapter, archived tokenizer, CPU predict probe', flush=True)
        return self

In [ ]:
class LegacyLoRAFeatures:
    @staticmethod
    @contextmanager
    def dense_peft():
        original_find = importlib.util.find_spec
        patched = []
        unavailable = lambda *args, **kwargs: False
        try:
            importlib.util.find_spec = lambda name, *args, **kwargs: None if name.split('.')[0] == 'torchao' else original_find(name, *args, **kwargs)
            peft = importlib.import_module('peft')
            utilities = importlib.import_module('peft.import_utils')
            original_check = utilities.is_torchao_available
            for name in ['peft.import_utils', 'peft.tuners.lora.model', 'peft.tuners.lora.torchao']:
                module = importlib.import_module(name)
                if hasattr(module, 'is_torchao_available'):
                    previous = module.is_torchao_available
                    patched.append((module, original_check if previous is unavailable else previous))
                    module.is_torchao_available = unavailable
            yield peft
        finally:
            for module, previous in reversed(patched):
                module.is_torchao_available = previous
            importlib.util.find_spec = original_find

    @staticmethod
    def probe_dense_peft():
        with LegacyLoRAFeatures.dense_peft() as peft:
            probe = nn.Sequential(nn.Linear(4, 4, bias=False, device='cpu'))
            configuration = peft.LoraConfig(r=2, lora_alpha=4, target_modules=['0'], lora_dropout=0.0, bias='none')
            adapted = peft.get_peft_model(probe, configuration).eval()
            with torch.inference_mode():
                result = adapted(torch.ones(1, 4, device='cpu'))
            if tuple(result.shape) != (1, 4) or not torch.isfinite(result).all():
                raise RuntimeError('CPU dense-LoRA compatibility probe failed')
        print('PEFT CPU DENSE-LORA PROBE PASSED: optional torchao dispatch disabled', flush=True)

    @staticmethod
    def check_anchors(actual, expected):
        if actual.shape != expected.shape or not np.isfinite(actual).all():
            raise ValueError('Invalid helper anchor feature matrix')
        relative = float(np.abs(actual - expected).mean() / (np.abs(expected).mean() + 1e-9))
        if relative >= 0.02:
            raise ValueError(f'Helper features do not match the pretrained bank anchors: mean relative error {relative:.5f}')
        return dict(mean_relative_error=relative, max_absolute_error=float(np.abs(actual - expected).max()), passed=True)

    @torch.no_grad()
    def batch(self, rows, ids, model, tokenizer, norms, output):
        device = model.get_input_embeddings().weight.device
        tokens, states = None, None
        try:
            tokens = tokenizer.pad({'input_ids': [ids[index] for index in rows]}, return_tensors='pt').to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                states = model(**tokens, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(states[layer]).all() for layer in HELPER_LAYERS):
                raise FloatingPointError('Non-finite legacy helper hidden states')
            mask = tokens['attention_mask'].unsqueeze(-1).float()
            mask[:, 0] = 0
            output[rows] = torch.cat([normalizer((states[layer].float() * mask.to(states[layer].device)).sum(1).to(device) / mask.sum(1).to(device))
                                      for layer, normalizer in zip(HELPER_LAYERS, norms)], -1).cpu().numpy()
        finally:
            del states, tokens

    def pool(self, texts, model, tokenizer, norms, label):
        ids = tokenizer(list(texts), add_special_tokens=False, truncation=True, max_length=447)['input_ids']
        ids = [tokens + [tokenizer.eos_token_id] for tokens in ids]
        order = np.argsort([len(tokens) for tokens in ids], kind='stable')
        output = np.zeros((len(texts), 32768), np.float32)
        for start in range(0, len(order), 8):
            rows = order[start:start + 8]
            try:
                self.batch(rows, ids, model, tokenizer, norms, output)
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                for row in rows:
                    self.batch([row], ids, model, tokenizer, norms, output)
            if start % 200 == 0 or start + 8 >= len(order):
                print(f'Legacy LoRA24 {label}: {min(start + 8, len(order))}/{len(order)} passages', flush=True)
        if not np.isfinite(output).all():
            raise FloatingPointError('Non-finite helper features')
        return output

    def extract(self, assets, texts):
        base = assets.helper_base
        tokenizer = AutoTokenizer.from_pretrained(str(assets.legacy / 'tokenizer'), local_files_only=True)
        tokenizer.padding_side = 'right'
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        configuration = AutoConfig.from_pretrained(str(base), local_files_only=True)
        configuration.num_hidden_layers, configuration.use_cache = 24, False
        if getattr(configuration, 'layer_types', None) is not None:
            configuration.layer_types = configuration.layer_types[:24]
        trunk = AutoModel.from_pretrained(str(base), config=configuration, dtype=torch.float16,
            device_map='auto', max_memory={0: '13GiB', 1: '13GiB'}, local_files_only=True).eval()
        if any(str(device) in {'cpu', 'disk'} for device in getattr(trunk, 'hf_device_map', {}).values()):
            del trunk
            raise RuntimeError('Legacy helper was offloaded to CPU/disk; refusing an unexpectedly slow run')
        model = None
        try:
            with self.dense_peft() as peft:
                model = peft.PeftModel.from_pretrained(trunk, assets.legacy / 'adapter').eval()
            device = model.get_input_embeddings().weight.device
            state = load_file(str(assets.legacy / 'heads.safetensors'))
            norms = nn.ModuleList([nn.LayerNorm(4096) for _ in HELPER_LAYERS]).to(device).float()
            norms.load_state_dict({key[len('norms.'):]: value for key, value in state.items() if key.startswith('norms.')})
            actual = self.pool(assets.anchor_texts, model, tokenizer, norms, 'anchors')
            parity = self.check_anchors(actual, assets.anchors['features'])
            print('HELPER GPU ANCHOR PARITY', json.dumps(parity), flush=True)
            features = self.pool(texts, model, tokenizer, norms, 'test')
            return features, parity
        finally:
            del model, trunk
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
class HelperContractChecks:
    @staticmethod
    def run():
        if sorted(HELPER_LAYERS) != HELPER_LAYERS or len(HELPER_LAYERS) != 8:
            raise AssertionError('Invalid legacy layer order')
        assert np.isclose(sum(HELPER_HEAD_WEIGHTS.values()), 1.0)
        reference = np.ones((3, 32768), dtype=np.float32)
        parity = LegacyLoRAFeatures.check_anchors(reference.copy(), reference)
        assert parity['passed'] is True and parity['mean_relative_error'] == 0.0
        json.dumps(parity)
        try:
            LegacyLoRAFeatures.check_anchors(reference + 0.1, reference)
        except ValueError:
            pass
        else:
            raise AssertionError('Mismatching legacy anchors were accepted')
        rng = np.random.RandomState(42)
        main = ProbabilityMath.softmax(rng.normal(size=(60, 18)))
        helper = ProbabilityMath.softmax(rng.normal(size=(60, 18)))
        mixed = 0.90 * main + 0.10 * helper
        ProbabilityMath.check(mixed, 60)
        ProbabilityMath.check(ProbabilityMath.balance(mixed), 60)

HelperContractChecks.run()


In [ ]:
assets = HelperInputs(config).preflight()
train, test, pool = assets.load_data()
assets.prepare_helper(train)
print(f'Train: {len(train)}, test: {len(test)}, public pool: {len(pool)}; bank: {config.helper_condition}')

In [ ]:
'''6. Source style classifier and average-linkage topic groups'''

In [ ]:
class StyleClassifier:
    @staticmethod
    def shape_text(text):
        return re.sub(r'[^\W\d_]+', lambda match: 'X' if match[0].isupper() else 'x', text)

    @staticmethod
    def function_text(text):
        return ' '.join(word.lower() if word.lower() in SMALL_WORDS or not word.isalnum() else 'CONTENT'
                        for word in re.findall(r'\w+|[^\w\s]', text))

    def fit_predict(self, train, test, pool):
        texts = train['text'].astype(str).tolist() + pool['text'].tolist()
        mapping = {label: index for index, label in enumerate(LABELS)}
        y = np.r_[train['model'].map(mapping).to_numpy(), pool['model'].map(mapping).to_numpy()]
        training, query = [], []
        for view, low, high, maximum, weight in VIEWS:
            prepare = self.shape_text if view == 'shape' else self.function_text if view == 'small' else str
            analyzer = 'char' if view == 'shape' else 'word' if view == 'small' else view
            options = dict(analyzer=analyzer, ngram_range=(low, high), max_features=maximum,
                           lowercase=False, sublinear_tf=True, min_df=2, dtype=np.float32)
            if analyzer == 'word':
                options['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
            vectorizer = TfidfVectorizer(**options)
            training.append(vectorizer.fit_transform([prepare(text) for text in texts]) * weight)
            query.append(vectorizer.transform([prepare(text) for text in test['text']]) * weight)
        classifier = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
        classifier.fit(hstack(training, format='csr'), y)
        if not np.array_equal(classifier.classes_, np.arange(len(LABELS))):
            raise ValueError('Unexpected SVC class order')
        return ProbabilityMath.softmax(classifier.decision_function(hstack(query, format='csr')))

In [ ]:
class TopicGrouping:
    @staticmethod
    def detect(texts):
        if len(texts) < 2:
            groups = np.ones(len(texts), dtype=int)
        else:
            features = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(list(texts))
            distance = np.clip(1 - (features @ features.T).toarray(), 0, None)
            np.fill_diagonal(distance, 0)
            groups = fcluster(linkage(squareform(distance, checks=False), method='average'), 0.85, criterion='distance')
        sizes = np.bincount(groups)[groups]
        return groups, (sizes >= 2) & (sizes <= 18)

In [ ]:
'''7. Source Part Two LoRA pooling
The main 14B extractors use training EOS 151643 and remove the final backbone norm.
Their current-test probabilities are computed exactly as in our Part Two baseline.
Training extraction is disabled in these two experiments.
The legacy helper is separate: archived EOS 151645 and original final norm retained.'''

In [ ]:
class LoRAFeatureExtractor:
    def __init__(self):
        self.token_audit = []

    def load_trunk(self, base, keep, dtype):
        cfg = AutoConfig.from_pretrained(str(base), local_files_only=True)
        cfg.num_hidden_layers, cfg.use_cache = keep, False
        if getattr(cfg, 'layer_types', None) is not None:
            cfg.layer_types = cfg.layer_types[:keep]
        hidden, intermediate, vocab = cfg.hidden_size, cfg.intermediate_size, cfg.vocab_size
        head_dim = getattr(cfg, 'head_dim', hidden // cfg.num_attention_heads)
        q, kv = cfg.num_attention_heads * head_dim, cfg.num_key_value_heads * head_dim
        gib = 2 * (vocab * hidden + keep * (2 * hidden * q + 2 * hidden * kv + 3 * hidden * intermediate)) / 2**30
        options = dict(config=cfg, dtype=dtype, local_files_only=True)
        if gib > 13:
            options.update(device_map='auto', max_memory={0: '13GiB', 1: '13GiB'})
            model = AutoModel.from_pretrained(str(base), **options).eval()
            if any(str(device) in {'cpu', 'disk'} for device in getattr(model, 'hf_device_map', {}).values()):
                raise RuntimeError('CPU/disk model offload detected; stop rather than run an unexpectedly slow model')
        else:
            model = AutoModel.from_pretrained(str(base), **options).to('cuda:0').eval()
        model.norm = nn.Identity()
        return model

    @torch.no_grad()
    def merge(self, model, directory):
        config = json.loads((directory / 'adapter/adapter_config.json').read_text())
        state = load_file(str(directory / 'adapter/adapter_model.safetensors'))
        modules = dict(model.named_modules())
        for name, left in state.items():
            if '.lora_A.' in name:
                prefix = name.split('.lora_A.')[0]
                layer = modules[prefix[prefix.index('layers.'):]]
                right = state[name.replace('.lora_A.', '.lora_B.')]
                delta = (right.float().to(layer.weight.device) @ left.float().to(layer.weight.device)) * (config['lora_alpha'] / config['r'])
                layer.weight.add_(delta.to(layer.weight.dtype))

    @torch.no_grad()
    def pool_batch(self, indices, ids, model, tokenizer, layers, norms, dtype, output):
        batch, hidden_states = None, None
        try:
            batch = tokenizer.pad({'input_ids': [ids[index] for index in indices]}, return_tensors='pt').to(model.embed_tokens.weight.device)
            hidden_states = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hidden_states[layer]).all() for layer in layers):
                hidden_states = None
                model.to(torch.bfloat16)
                hidden_states = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
                if not all(torch.isfinite(hidden_states[layer]).all() for layer in layers):
                    raise FloatingPointError('Non-finite hidden states after the bfloat16 retry')
            mask = batch['attention_mask'].unsqueeze(-1).float().to('cuda:0')
            mask[:, 0] = 0
            denominator = mask.sum(1).clamp(min=1)
            output[indices] = torch.cat([normalizer((hidden_states[layer].to('cuda:0').float() * mask).sum(1) / denominator)
                                        for layer, normalizer in zip(layers, norms)], -1).cpu().numpy()
        finally:
            del hidden_states, batch

    def pool(self, texts, model, tokenizer, settings, norms, dtype, batch_size, name):
        ids = tokenizer(list(texts), add_special_tokens=False, truncation=True, max_length=settings['max_len'])['input_ids']
        ids = [tokens + [settings['eos_id']] for tokens in ids]
        order = np.argsort([len(tokens) for tokens in ids], kind='stable')
        output = np.zeros((len(ids), model.config.hidden_size * len(settings['layers'])), np.float32)
        for start in range(0, len(order), batch_size):
            indices = order[start:start + batch_size]
            try:
                self.pool_batch(indices, ids, model, tokenizer, settings['layers'], norms, dtype, output)
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                for index in indices:
                    self.pool_batch([index], ids, model, tokenizer, settings['layers'], norms, dtype, output)
            if start % (25 * batch_size) == 0 or start + batch_size >= len(order):
                print(f'{name}: {min(start + batch_size, len(order))}/{len(order)} passages', flush=True)
        if not np.isfinite(output).all():
            raise FloatingPointError(f'Invalid pooled features: {name}')
        return output

    def extract(self, directory, base, test_texts, train_texts, batch_size):
        settings = json.loads((directory / 'meta.json').read_text())['cfg']
        tokenizer = AutoTokenizer.from_pretrained(str(base), local_files_only=True)
        tokenizer.padding_side = 'right'
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.token_audit.append(dict(checkpoint=directory.name, training_eos_id=settings['eos_id'], tokenizer_eos_id=tokenizer.eos_token_id))
        print(f"EOS {directory.name}: training={settings['eos_id']}, tokenizer={tokenizer.eos_token_id}", flush=True)
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        model = self.load_trunk(base, settings['keep'], dtype)
        try:
            self.merge(model, directory)
            heads = load_file(str(directory / 'heads.safetensors'))
            hidden = model.config.hidden_size
            norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in settings['layers']]).to('cuda:0').float()
            norms.load_state_dict({key[len('norms.'):]: value for key, value in heads.items() if key.startswith('norms.')})
            query = self.pool(test_texts, model, tokenizer, settings, norms, dtype, batch_size, directory.name + ' test')
            training = (self.pool(train_texts, model, tokenizer, settings, norms, dtype, batch_size, directory.name + ' train')[:, -hidden:].copy()
                        if train_texts is not None else None)
            readouts = {key: value.float().numpy() for key, value in heads.items() if not key.startswith('norms.')}
            return query, training, readouts
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
'''8. Fully fine-tuned DeBERTa
Stage the encoder locally before loading to avoid many reads from the dataset mount.'''

In [ ]:
class DebertaClassifier:
    @torch.no_grad()
    def predict(self, directory, texts):
        metadata = json.loads((directory / 'meta.json').read_text())
        tokenizer = AutoTokenizer.from_pretrained(str(directory / 'encoder'), local_files_only=True)
        model = AutoModel.from_pretrained(str(directory / 'encoder'), dtype=torch.float32, local_files_only=True).to('cuda:0').eval()
        try:
            head = load_file(str(directory / 'head.safetensors'))
            ids = tokenizer(list(texts), truncation=True, max_length=metadata['max_len'])['input_ids']
            order = np.argsort([len(tokens) for tokens in ids], kind='stable')
            output = np.zeros((len(ids), model.config.hidden_size), np.float32)
            for start in range(0, len(order), 32):
                rows = order[start:start + 32]
                batch = tokenizer.pad({'input_ids': [ids[index] for index in rows]}, return_tensors='pt').to('cuda:0')
                mask = batch['attention_mask'].unsqueeze(-1).float()
                states = model(**batch).last_hidden_state
                output[rows] = ((states * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()
                del batch, mask, states
                print(f'DeBERTa: {min(start + 32, len(order))}/{len(order)} passages', flush=True)
            return ProbabilityMath.softmax(output @ head['weight'].float().numpy().T + head['bias'].float().numpy())
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
'''9. Submission and experiment orchestration'''

In [ ]:
class SubmissionWriter:
    def __init__(self, root):
        self.root = root

    def write(self, test, probabilities, name='submission.parquet'):
        ProbabilityMath.check(probabilities, len(test))
        frame = pd.DataFrame({'id': test['id'].astype(str), 'model': np.asarray(LABELS)[probabilities.argmax(1)]})
        if frame['id'].isna().any() or frame['id'].duplicated().any() or not frame['model'].isin(LABELS).all():
            raise ValueError('Invalid submission IDs or model labels')
        path = self.root / name
        frame.to_parquet(path, index=False)
        pd.testing.assert_frame_equal(frame, pd.read_parquet(path))
        return frame

In [ ]:
class HelperExperiment:
    def __init__(self, config, assets, train, test, pool):
        self.config, self.assets = config, assets
        self.train, self.test, self.pool = train, test, pool
        self.extractor = LoRAFeatureExtractor()
        self.timer = StageTimer(config.output_root)

    def run(self):
        if (self.config.output_root / 'submission.parquet').exists():
            raise FileExistsError('Start a clean run; submission.parquet already exists')
        if torch.cuda.device_count() != 2 or any('T4' not in torch.cuda.get_device_name(index) for index in range(2)):
            raise RuntimeError('Select GPU T4 x2; use run_inference=False for CPU-only input checks')
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        started = time.monotonic()
        try:
            with threadpool_limits(limits=4):
                return self.predict(started)
        except BaseException:
            path = self.config.output_root / 'submission.parquet'
            if path.exists():
                path.rename(path.with_name('failed_submission.parquet'))
            raise

    def predict(self, started):
        train, test = self.train, self.test
        members = {}
        with self.timer.measure('Part Two SVC on pool_v4'):
            probabilities = StyleClassifier().fit_predict(train, test, self.pool)
            ProbabilityMath.check(probabilities, len(test))
            logits = np.log(np.clip(probabilities, 1e-9, 1))
            members['svc'] = probabilities
        del self.pool
        groups, eligible = TopicGrouping.detect(test['text'].tolist())
        for key, settings in self.config.loras.items():
            with self.timer.measure(key + ': staging and test-only inference'):
                directory = LocalCheckpointStage.copy(self.assets.assets / key, self.config.scratch_root / key)
                features, _, heads = self.extractor.extract(directory, self.assets.bases[key], test['text'].tolist(), None, settings['bs'])
                plain = ProbabilityMath.softmax(features @ heads['plain.weight'].T + heads['plain.bias'])
                ProbabilityMath.check(plain, len(test))
                logits += settings['w'] * np.log(np.clip(plain, 1e-9, 1))
                members[key] = plain
                if eligible.any():
                    centered = features.copy()
                    for group in np.unique(groups[eligible]):
                        centered[groups == group] -= centered[groups == group].mean(0)
                    twin = ProbabilityMath.softmax(centered[eligible] @ heads['centered.weight'].T + heads['centered.bias'])
                    logits[eligible] += settings['twin_w'] * np.log(np.clip(twin, 1e-9, 1))
                    full = np.full_like(plain, np.nan)
                    full[eligible] = twin
                    members[key + '_twin'] = full
                    del centered
                del features, heads
        with self.timer.measure('Fine-tuned DeBERTa'):
            directory = LocalCheckpointStage.copy(self.assets.assets / 'deberta_v3_large', self.config.scratch_root / 'deberta_v3_large')
            probabilities = DebertaClassifier().predict(directory, test['text'].tolist())
            ProbabilityMath.check(probabilities, len(test))
            logits += 0.25 * np.log(np.clip(probabilities, 1e-9, 1))
            members['deberta'] = probabilities
        baseline = ProbabilityMath.softmax(logits)
        writer = SubmissionWriter(self.config.output_root)
        baseline_labels = writer.write(test, ProbabilityMath.balance(baseline), 'submission_baseline.parquet')
        np.savez_compressed(self.config.output_root / 'baseline_probabilities.npz', ids=test['id'].to_numpy(dtype=str),
            labels=np.asarray(LABELS, dtype=str), probabilities=baseline)
        with self.timer.measure('Exact legacy LoRA24 anchors and current-test features'):
            features, parity = LegacyLoRAFeatures().extract(self.assets, test['text'].tolist())
            np.save(self.config.output_root / 'helper_current_test_features.npy', features, allow_pickle=False)
        with self.timer.measure('Public bank CPU prediction and fixed blend'):
            helper, heads = self.assets.public_bank.predict(features, test['text'].tolist())
            np.savez_compressed(self.config.output_root / 'helper_probabilities.npz', ids=test['id'].to_numpy(dtype=str),
                labels=np.asarray(LABELS, dtype=str), probabilities=helper, **heads)
            selected_raw = (1 - self.config.helper_weight) * baseline + self.config.helper_weight * helper
            selected = ProbabilityMath.balance(selected_raw)
            ProbabilityMath.check(selected, len(test))
            np.savez_compressed(self.config.output_root / 'enhanced_probabilities.npz', ids=test['id'].to_numpy(dtype=str),
                labels=np.asarray(LABELS, dtype=str), baseline_raw=baseline, helper=helper, selected_raw=selected_raw,
                probabilities=selected, groups=groups, eligible=eligible, **members)
            selected_labels = np.asarray(LABELS)[selected.argmax(1)]
            changed = int((selected_labels != baseline_labels['model'].to_numpy()).sum())
            report = dict(mode='part_two_public_readout', helper_condition=self.config.helper_condition,
                helper_weight=float(self.config.helper_weight), synthetic_rows=int(self.assets.public_bank.bank['synthetic_rows']),
                source_url=SOURCE_URL, helper_source=HELPER_BANK_URL, original_helper_author='kirilltym',
                source_sha256=READOUT_SOURCE_SHA256, adapter_sha256=HELPER_ADAPTER_SHA256,
                bank_sha256=self.assets.public_bank.record['readout_sha256'],
                gpu_anchor_parity=parity, input_csv_sha256=self.assets.data_hashes,
                rows=int(len(test)), changed_labels_vs_baseline=changed,
                elapsed_seconds=float(time.monotonic() - started),
                sklearn_version_warnings=self.assets.public_bank.compatibility_warnings,
                score_reproduced=False, all_required_members_completed=True,
                no_training_feature_extraction=True, no_hidden_prediction_cache_reuse=True)
            (self.config.output_root / 'experiment_audit.json').write_text(json.dumps(report, indent=2))
            submission = writer.write(test, selected)
            print(json.dumps({key: report[key] for key in ['helper_condition', 'changed_labels_vs_baseline', 'elapsed_seconds']}, indent=2), flush=True)
            if not changed:
                print('No labels changed on this pool. This output duplicates the baseline on these IDs.', flush=True)
        return submission

In [ ]:
'''10. Execute
Use a fresh GPU T4 x2 session. Every required model must finish; no helper is silently skipped.
The baseline submission and probabilities are saved before the extra helper model runs.
The selected output is submission.parquet. A CPU-only input-check version cannot be submitted.
The GPU anchor check must pass before current-test helper inference.
Format checks and CPU probes do not establish hidden-test runtime or leaderboard accuracy.'''

In [ ]:
if config.run_inference:
    experiment = HelperExperiment(config, assets, train, test, pool)
    submission = experiment.run()
    print(submission.head())
else:
    print('CPU INPUT CHECKS COMPLETE: no model inference and no submission file produced.')

## Credits

Public passages, CC BY 4.0 unless another license is given.

- [seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages)
- [willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest)
- [lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1)
- [prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
- [alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations)
- [haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages)
- [kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages)
- [lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations)
- [abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations)
- [dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations)
- [ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations)
- [keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1)
- [michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1)
- [jadkutbi/ntl-own-proxy-generations](https://www.kaggle.com/datasets/jadkutbi/ntl-own-proxy-generations)
- [pardheev/ntl-proxy-extra-passages](https://www.kaggle.com/datasets/pardheev/ntl-proxy-extra-passages)
- [abhinavsaipodugu/ntl-proxy-gpt-gemini-passages](https://www.kaggle.com/datasets/abhinavsaipodugu/ntl-proxy-gpt-gemini-passages)
- [zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations](https://www.kaggle.com/datasets/zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations)
- [zbigniewkirchner/ntl-extra-gpt-passages](https://www.kaggle.com/datasets/zbigniewkirchner/ntl-extra-gpt-passages)
- [arxitekton/ntl-claude-gen-v1](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v1)
- [arxitekton/ntl-claude-gen-v2](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v2)
- [maniceet/name-that-llm-openrouter-openai-generations](https://www.kaggle.com/datasets/maniceet/name-that-llm-openrouter-openai-generations)
- [dedruck/ntl-proxy-gen](https://www.kaggle.com/datasets/dedruck/ntl-proxy-gen)
- [prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1), CC BY-SA 4.0
- [drjcfmd/ntl-generated-v2](https://www.kaggle.com/datasets/drjcfmd/ntl-generated-v2), CC BY-SA 4.0
- [illiadolenko/ntl-generated-corpus-v1](https://www.kaggle.com/datasets/illiadolenko/ntl-generated-corpus-v1), CC0
- [aswathvanan/ntl-proxy-passages](https://www.kaggle.com/datasets/aswathvanan/ntl-proxy-passages), CC BY-SA 4.0, only in stage one of the Qwen3-14B LoRA

Ideas and recipes.

- The LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7)
- The style aware n-gram views from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
- Adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering)

Models. Qwen3-14B-Base and Qwen2.5-14B by the Qwen team, Apache 2.0. DeBERTa-v3-large by Microsoft, MIT.

## Artifacts and validation status

`submission_baseline.parquet` and `baseline_probabilities.npz` are written before helper inference. `helper_probabilities.npz` includes each of the eight head predictions. `enhanced_probabilities.npz` stores ID-aligned raw main/helper/selected probabilities for CPU comparisons on this exact pool. `helper_current_test_features.npy` stores the actual 32768-dimensional legacy features. `helper_input_manifest.json` records dependency checks, and `experiment_audit.json` records the GPU anchor result and changed labels.

No stored test predictions are loaded during scoring. A three-row visible artifact is not a hidden-test cache. A zero-change visible run says nothing about changes on the hidden pool.

The real public banks were CPU-probed on published saved features. Full transformer loading, PEFT inference, GPU anchor parity, hidden runtime and competition scores remain unverified. No Kaggle GPU run was launched.